# Video Tokenizer Training (Spacetime Vector Quantized Variational Autoencoder)

In [1]:
import torch 
import lpips

from torch.utils.data import DataLoader
from torchvision.datasets import UCF101

import lightning as L
from spacetime.models.st_vq_vae import STVQVae

import wandb

In [2]:
wandb.login()

wandb: Currently logged in as: aryaman-pandya-wayve (wayve-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Lightning module

We will use pytorch lightning to reduce boiler plate (there's a lot in previous notebooks, despite the centralized modules in `/src`)

In [3]:
class STVQVaeModule(L.LightningModule):
    def __init__(
        self,
        num_heads,
        d_model,
        num_layers,
        d_linear,
        codebook_size,
        codebook_dim,
        patch_size,
        frame_height,
        frame_width,
        num_frames,
        num_linear_layers=2,
        num_groups=8,
        dropout=0.1,
        beta=0.1
    ):
        super().__init__()
        self.model = STVQVae(
            num_heads=num_heads,
            d_model=d_model,
            num_layers=num_layers,
            d_linear=d_linear,
            codebook_size=codebook_size,
            codebook_dim=codebook_dim,
            patch_size=patch_size,
            frame_height=frame_height,
            frame_width=frame_width,
            num_frames=num_frames,
            num_linear_layers=num_linear_layers,
            num_groups=num_groups,
            dropout=dropout
        )
        self.beta = beta
        self.example_clip = None
        self.example_recon = None

        self.lpips_metric = lpips.LPIPS(net="vgg")
        self.lpips_metric.eval()
        for p in self.lpips_metric.parameters():
            p.requires_grad = False

    def forward(self, inputs):
        return self.model(inputs)
    
    def configure_optimizers(self):
        return torch.optim.AdamW(self.model.parameters(), lr=3e-4)

    def training_step(self, batch, batch_idx):
        x, _ = batch
        x_pred, z_e, z_quantized = self(x)
        recon_loss = torch.nn.functional.mse_loss(x_pred, x)
        commit_loss = torch.nn.functional.mse_loss(z_e, z_quantized.detach())
        loss = recon_loss + (self.beta * commit_loss)

        self._log_losses(loss, recon_loss, commit_loss, is_training=True)

        if batch_idx == 0:
            self.example_clip = x[:1].detach().cpu()
            self.example_recon = x_pred[:1].detach().cpu()
        return loss

    def validation_step(self, batch, batch_idx):
        x, _ = batch
        x_pred, z_e, z_quantized = self(x)
        recon_loss = torch.nn.functional.mse_loss(x_pred, x)
        commit_loss = torch.nn.functional.mse_loss(z_e, z_quantized.detach())
        loss = recon_loss + (self.beta * commit_loss)

        self._log_losses(loss, recon_loss, commit_loss, is_training=False)

        with torch.no_grad():
        # LPIPS expects inputs in [-1,1]; convert if you’re in [0,1]
            B, C, F, H, W = x.shape
            to_lpips = lambda t: ((t * 2.0) - 1.0).reshape(B * F, C, H, W)
            lpips_val = self.lpips_metric(to_lpips(x_pred), to_lpips(x)).mean()
        self.log("val_lpips", lpips_val, prog_bar=False, logger=True)
        if wandb.run is not None:
            wandb.log({"val_lpips": lpips_val.item()}, step=self.global_step)
        return loss
    
    def on_validation_epoch_end(self):
        if self.example_clip is None or wandb.run is None:
            return
        clip = (self.example_clip.clamp(0, 1) * 255).to(torch.uint8)
        recon = (self.example_recon.clamp(0, 1) * 255).to(torch.uint8)
        video = torch.cat([clip, recon], dim=4)       # or dim=2/3, whichever you chose
        video = video.squeeze(0).permute(1, 0, 2, 3)  # (F, C, H, W)
        wandb.log(
            {
                "recon_video": wandb.Video(
                    video.squeeze(0), fps=4, format="mp4"
                )
            },
            step=self.global_step,
        )
        self.example_clip = None
        self.example_recon = None
    
    def _log_losses(self, loss, recon_loss, commit_loss, is_training=True):
        prefix = "train" if is_training else "val"
        log_on_step = True if is_training else False
        log_on_epoch = True

        # Lightning logging
        self.log(f"{prefix}_loss", loss, on_step=log_on_step, on_epoch=log_on_epoch, prog_bar=True, logger=True)
        self.log(f"{prefix}_recon_loss", recon_loss, on_step=log_on_step, on_epoch=log_on_epoch, prog_bar=False, logger=True)
        self.log(f"{prefix}_commit_loss", commit_loss, on_step=log_on_step, on_epoch=log_on_epoch, prog_bar=False, logger=True)

        # Weights & Biases logging
        if wandb.run is not None:
            wandb.log({
                f"{prefix}_loss": loss.item(),
                f"{prefix}_recon_loss": recon_loss.item(),
                f"{prefix}_commit_loss": commit_loss.item(),
            }, step=self.global_step)

    

## ProcGen: Heist Dataset Handling 

We will train our model on the "heist" environment from the [OpenAI Procgen Benchmark](https://github.com/openai/procgen). 
- Procgen is a suite of procedurally generated environments designed for benchmarking generalization in reinforcement learning agents.
- The Heist environment features randomly generated maze layouts on each episode, requiring the agent to navigate and collect keys to unlock safes.
- The dataset is generated by running the script in `src/spacetime/scripts/gen_procgen_heist.py`
- Our dataset consists of frame sequences and corresponding agent actions collected from Heist, preprocessed and saved as `.npz` shards for efficient loading.

In [10]:
from spacetime.utils.data import ProcgenShardDataset
from pathlib import Path

shard_dir = Path("../data/procgen_heist/shards")
shard_dir.mkdir(parents=True, exist_ok=True)

shard_dataset = ProcgenShardDataset(shard_dir, normalize=True)

In [11]:
from torch.utils.data import random_split

train_ratio = 0.8

train_size = int(train_ratio * len(shard_dataset))
val_size = len(shard_dataset) - train_size

train_dataset, val_dataset = random_split(
    shard_dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

In [12]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
)


## Tokenizer Hyperparameters 

In [13]:
params = {
    "num_heads": 4,  # starting with 4 heads paper uses 8 
    "d_model": 512,
    "num_layers": 4,  # starting with 4 layers paper uses 8
    "d_linear": 1536,
    "codebook_size": 1024,   # match latent model's num_discrete_actions
    "codebook_dim": 32,
    "patch_size": 8,  # starting with 8 paper uses 4 (but in a different type of setup)
    "frame_height": 64,
    "frame_width": 64,
    "num_frames": 16,
    "num_linear_layers": 2,
    "num_groups": 8,
    "dropout": 0.3,
    "max_epochs": 10,
    "precision": 16,
}

wandb.init(
    project="spacetime",
    name=f"stvqvae_ema_layers{params['num_layers']}_codebook_dim{params['codebook_dim']}_codebook_size{params['codebook_size']}_heads{params['num_heads']}_beta.1",
    config=params,
)

In [14]:

lightning_timesformer = STVQVaeModule(
    num_heads=params["num_heads"],
    d_model=params["d_model"],
    num_layers=params["num_layers"],
    d_linear=params["d_linear"],
    codebook_size=params["codebook_size"],
    codebook_dim=params["codebook_dim"],
    patch_size=params["patch_size"],
    frame_height=params["frame_height"],
    frame_width=params["frame_width"],
    num_frames=params["num_frames"],
    num_linear_layers=params["num_linear_layers"],
    num_groups=params["num_groups"],
    dropout=params["dropout"],
)

wandb.watch(lightning_timesformer, log="gradients", log_freq=100)

# trainer = L.Trainer(max_epochs=5, precision=32)


# sanity check: 

trainer = L.Trainer(
    max_epochs=1,
    limit_train_batches=1,
    limit_val_batches=1,
    fast_dev_run=False,
)

trainer.fit(model=lightning_timesformer, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

wandb.finish()

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Trainer will use only 1 of 4 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=4)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
💡 Tip: For seamless cloud uploads and versioning, try

Loading model from: /home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/lpips/weights/v0.1/vgg.pth


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
`Trainer(limit_train_batches=1)` was configured so 1 batch per epoch will be used.
`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name         | Type    | Params | Mode 
-------------------------------------------------
0 | model        | STVQVae | 80.2 M | train
1 | lpips_metric | LPIPS   | 14.7 M | eval 
-------------------------------------------------
80.2 M    Trainable params
14.7 M    Non-trainable params
94.9 M    Total params
379.545   Total estimated model params size (MB)
220       Modules in train mode
59        Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:527: Found 59 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/imageio_ffmpeg/_utils.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
`Trainer.fit` stopped: `max_epochs=1` reached.


train_commit_loss,▁
train_loss,▁
train_recon_loss,▁
val_commit_loss,▁█
val_loss,█▁
val_lpips,█▁
val_recon_loss,█▁
train_commit_loss,0.52433
train_loss,0.89532
train_recon_loss,0.84289
val_commit_loss,2.08456
